# Notebook 17R — Results Assembly (all 13 tables, every field)

**Abhishek Tomar | PN1196973 | LJMU | April 2026**

---

This notebook does **no training**. It reads the saved prediction files and the
existing result CSVs, then computes and assembles **every field of every table** —
including the fields that were computed but never printed (Precision, Recall,
Cohen Kappa) and the one genuinely new analytical layer: **per-class
(negative / neutral / positive) precision, recall and F1**, overall and per subgroup.

Design decisions baked in:
- **Four linguistic subgroups** (emoji-heavy, slang-heavy, sarcasm, formal). Posts
  matching multiple informal rules are assigned to their highest-priority single
  subgroup; no separate 'mixed' group (it would fragment the small subgroups and
  produce uninterpretable contrasts against the formal reference).
- Sarcasm (n=14) is reported but flagged **exploratory**.

Outputs: one clean CSV per table into `results/assembled/` — **all 13 tables in one
place**, ready to paste into the tracking sheet, plus a per-class table (Table 1c) that
is new.

**Run NB7T first** (it tunes all five baselines and saves the full Table 3). This
notebook then reads that file so Table 3 is the complete five-model version. If NB7T
has not been run, Table 3 falls back to the single-model Exp 10 result and tells you so.


## Cell 1: Setup

In [1]:
!pip install -q emoji xgboost scikit-learn pandas pyarrow

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append(r"/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation")
from thesis_utils import *

import pandas as pd, numpy as np, json
from pathlib import Path
from sklearn.metrics import (precision_recall_fscore_support, precision_score,
                             recall_score, f1_score, accuracy_score,
                             matthews_corrcoef, cohen_kappa_score, confusion_matrix)

OUT = PATHS["results"] / "assembled"
OUT.mkdir(parents=True, exist_ok=True)
print("assembled output dir:", OUT)

CLASSES = ["negative", "neutral", "positive"]
SUBGROUPS_4 = ["emoji-heavy", "slang-heavy", "sarcasm", "formal"]
REF = "formal"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 3.5 MB/s eta 0:00:00
Mounted at /content/drive
thesis_utils loaded. [V2 FIXED: Fairlearn EOD, Theil index]
  seed = 42
  subgroup thresholds: emoji > 0.05, slang > 0.1, formal min words = 8
  reference subgroup  = formal
assembled output dir: /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation/results/assembled


## Cell 2: Helper — the complete metric block

One function that returns EVERY metric for a (y_true, y_pred, y_proba) triple:
overall aggregates AND per-class precision/recall/F1. This is the piece that was
missing — per-class numbers were never surfaced anywhere.

In [2]:
def full_metric_block(yt, yp, proba=None, label=""):
    yt = np.asarray(yt); yp = np.asarray(yp)
    row = {
        "Label": label,
        "N": len(yt),
        "Accuracy": accuracy_score(yt, yp),
        "Precision (macro)": precision_score(yt, yp, average="macro", zero_division=0),
        "Recall (macro)": recall_score(yt, yp, average="macro", zero_division=0),
        "Macro F1": f1_score(yt, yp, average="macro", zero_division=0),
        "Weighted F1": f1_score(yt, yp, average="weighted", zero_division=0),
        "MCC": matthews_corrcoef(yt, yp) if len(set(yt)) > 1 else np.nan,
        "Cohen Kappa": cohen_kappa_score(yt, yp) if len(set(yt)) > 1 else np.nan,
    }
    if proba is not None:
        row["Brier"] = multiclass_brier(yt, proba)
    # per-class
    p, r, f, s = precision_recall_fscore_support(
        yt, yp, labels=CLASSES, average=None, zero_division=0)
    for i, c in enumerate(CLASSES):
        row[f"{c.capitalize()} Precision"] = p[i]
        row[f"{c.capitalize()} Recall"]    = r[i]
        row[f"{c.capitalize()} F1"]        = f[i]
        row[f"{c.capitalize()} Support"]   = int(s[i])
    return row

def proba_cols(df):
    cols = sorted([c for c in df.columns if c.startswith("proba_")])
    return df[cols].values if cols else None

print("metric block ready")


metric block ready


## Cell 3: TABLE 1 — Baselines (complete: adds Precision, Recall, Cohen Kappa, per-class)

In [3]:
MODELS = [("exp01","LogisticRegression"),("exp02","LinearSVM"),
          ("exp03","MultinomialNB"),("exp04","RandomForest"),("exp05","XGBoost")]

rows = []
for exp_id, name in MODELS:
    p = load_predictions(exp_id, name)
    row = full_metric_block(p["y_true"], p["y_pred"], proba_cols(p), label=name)
    row["Dataset"] = "TweetEval"; row["Feature Used"] = "TF-IDF unigram + bigram"
    wga, worst = worst_group_accuracy(p); row["WGA"] = wga; row["Worst Subgroup"] = worst
    rows.append(row)

table1 = pd.DataFrame(rows).sort_values("Macro F1", ascending=False).reset_index(drop=True)
table1["Best / Not Best"] = ["BEST"] + ["Not Best"]*(len(table1)-1)

order = ["Label","Dataset","Feature Used","Accuracy","Precision (macro)","Recall (macro)",
         "Macro F1","Weighted F1","MCC","Cohen Kappa","Brier","WGA","Worst Subgroup","Best / Not Best",
         "Negative Precision","Negative Recall","Negative F1",
         "Neutral Precision","Neutral Recall","Neutral F1",
         "Positive Precision","Positive Recall","Positive F1"]
table1 = table1[[c for c in order if c in table1.columns]].rename(columns={"Label":"Model"})
table1.round(4).to_csv(OUT/"Table1_complete.csv", index=False)
print(table1[["Model","Accuracy","Precision (macro)","Recall (macro)","Macro F1",
              "Negative Recall","Negative F1"]].round(4).to_string(index=False))


             Model  Accuracy  Precision (macro)  Recall (macro)  Macro F1  Negative Recall  Negative F1
LogisticRegression    0.5800             0.5738          0.5782    0.5672           0.4549       0.5299
         LinearSVM    0.5876             0.5896          0.5727    0.5648           0.3847       0.4879
     MultinomialNB    0.5791             0.5787          0.5631    0.5589           0.4021       0.4949
           XGBoost    0.5542             0.6305          0.4775    0.4519           0.1128       0.1968
      RandomForest    0.5394             0.6516          0.4688    0.4169           0.0388       0.0742


## Cell 4: TABLE 1c — Per-class performance of the FINAL model (NEW)

The single most valuable addition. The negative class is only 19% of the data, so
per-class negative performance is the number the imbalance/fairness story hinges on,
and it was never in any table.

In [4]:
pred = load_predictions("exp10", "FinalModel")
p, r, f, s = precision_recall_fscore_support(
    pred["y_true"], pred["y_pred"], labels=CLASSES, average=None, zero_division=0)

table1c = pd.DataFrame({
    "Class": CLASSES,
    "Precision": p, "Recall": r, "F1": f, "Support": s.astype(int),
    "Support %": (s/s.sum()*100).round(1),
})
table1c.round(4).to_csv(OUT/"Table1c_final_perclass.csv", index=False)
print("PER-CLASS — FINAL MODEL")
print(table1c.round(4).to_string(index=False))
print("\nNote: negative is the minority class; its recall is the key imbalance metric.")


PER-CLASS — FINAL MODEL
   Class  Precision  Recall     F1  Support  Support %
negative     0.6487  0.4877 0.5568     3972       32.3
 neutral     0.6204  0.6459 0.6329     5937       48.3
positive     0.5053  0.6632 0.5736     2375       19.3

Note: negative is the minority class; its recall is the key imbalance metric.


## Cell 5: TABLE 2 — Feature comparison (already complete in CSV; re-assembled with per-subgroup F1)

In [5]:
feat = [("exp06ref","TF-IDF unigram + bigram (reference)"),
        ("exp07","Character n-grams (3-5)"),
        ("exp08","TF-IDF + social media features"),
        ("exp09","Hybrid feature set")]
rows=[]
for exp_id, fname in feat:
    p = load_predictions(exp_id, "LogisticRegression")
    row = full_metric_block(p["y_true"], p["y_pred"], proba_cols(p), label=fname)
    row["Experiment No."] = exp_id
    row["Error Rate"] = 1 - row["Accuracy"]
    rep = subgroup_report(p)
    for _, rr in rep.iterrows():
        row[f"{rr['Subgroup']} F1"] = rr["Macro F1"]
    rows.append(row)
table2 = pd.DataFrame(rows)
best = table2["Macro F1"].idxmax()
table2["Selected for Final Model?"] = ["YES" if i==best else "No" for i in table2.index]
cols2 = ["Experiment No.","Label","Accuracy","Macro F1","Weighted F1","Error Rate",
         "Negative F1","Neutral F1","Positive F1",
         "formal F1","other F1","emoji-heavy F1","slang-heavy F1","sarcasm F1",
         "Selected for Final Model?"]
table2 = table2[[c for c in cols2 if c in table2.columns]].rename(columns={"Label":"Feature Set"})
table2.round(4).to_csv(OUT/"Table2_complete.csv", index=False)
print(table2[["Experiment No.","Macro F1","Negative F1","Selected for Final Model?"]].round(4).to_string(index=False))


Experiment No.  Macro F1  Negative F1 Selected for Final Model?
      exp06ref    0.5453       0.4013                        No
         exp07    0.5846       0.4889                        No
         exp08    0.5439       0.4006                        No
         exp09    0.5945       0.5209                       YES


## Cell 6: TABLE 4 — Subgroup performance (complete: adds per-class within each subgroup)

Four subgroups. Each row now carries the standard block PLUS negative/neutral/positive
recall within that subgroup — so you can see, e.g., how sarcasm handles negative
sentiment specifically. Bootstrap CI on macro F1 retained.

In [6]:
rows=[]
for sg in SUBGROUPS_4:
    part = pred[pred["subgroup"]==sg]
    if len(part)==0: continue
    yt, yp = part["y_true"].values, part["y_pred"].values
    row = full_metric_block(yt, yp, proba_cols(part), label=sg)
    # bootstrap CI on macro F1
    _, lo, hi = bootstrap_ci(yt, yp, n_boot=1000)
    row["Macro F1 CI Low"]=lo; row["Macro F1 CI High"]=hi
    # FPR/FNR
    cm = confusion_matrix(yt, yp, labels=CLASSES)
    with np.errstate(divide="ignore", invalid="ignore"):
        fp=cm.sum(0)-np.diag(cm); fn=cm.sum(1)-np.diag(cm)
        tp=np.diag(cm); tn=cm.sum()-(fp+fn+tp)
        row["FPR"]=float(np.nanmean(fp/(fp+tn))); row["FNR"]=float(np.nanmean(fn/(fn+tp)))
    row["Misclassification Rate"]=1-row["Accuracy"]
    rows.append(row)

table4 = pd.DataFrame(rows)
ref_f1 = table4.loc[table4["Label"]==REF,"Macro F1"].iloc[0]
table4["F1 Gap"] = table4["Macro F1"] - ref_f1
def risk(g):
    if g <= -0.10: return "Low (informal ahead)"
    if abs(g) < 0.05: return "Low"
    return "Medium" if abs(g)<0.10 else "High"
table4["Risk Level"] = table4.apply(
    lambda r: "Reference" if r["Label"]==REF else risk(r["F1 Gap"]), axis=1)
table4.loc[table4["Label"]=="sarcasm","Risk Level"] = "Low (exploratory, n=14)"

cols4 = ["Label","N","Accuracy","Macro F1","Macro F1 CI Low","Macro F1 CI High",
         "FPR","FNR","Misclassification Rate","F1 Gap","Risk Level",
         "Negative Precision","Negative Recall","Negative F1",
         "Neutral Precision","Neutral Recall","Neutral F1",
         "Positive Precision","Positive Recall","Positive F1"]
table4 = table4[[c for c in cols4 if c in table4.columns]].rename(
    columns={"Label":"Subgroup","N":"Number of Samples"})
table4.round(4).to_csv(OUT/"Table4_complete.csv", index=False)
print(table4[["Subgroup","Number of Samples","Macro F1","F1 Gap","Negative Recall","Risk Level"]].round(4).to_string(index=False))


   Subgroup  Number of Samples  Macro F1  F1 Gap  Negative Recall              Risk Level
emoji-heavy                696    0.6022  0.0221           0.4658                     Low
slang-heavy                 60    0.6180  0.0378           0.5882                     Low
    sarcasm                 14    0.7897  0.2096           0.8000 Low (exploratory, n=14)
     formal              10398    0.5801  0.0000           0.4924               Reference


## Cell 7: TABLE 5 — Misclassification patterns (complete: real counts + example texts)

Fills the sheet's planned rows including the sarcasm and slang error rows that DO
exist in the data, and pulls Example Text from the saved LIME/confident-error file.

In [7]:
errors = pred[pred["correct"]==0].copy()
total = len(errors)

# attach text if available
D = PATHS["data"]
try:
    tw = pd.read_parquet(D/"tw_test.parquet").reset_index(drop=True)
    errors = errors.reset_index(drop=True)
    # align by position within the saved prediction order is not guaranteed;
    # so instead read the saved misclassified_instances which already has text
    mis = pd.read_parquet(D/"misclassified_instances.parquet")
    text_lookup = mis.set_index([mis["subgroup"], mis["y_true"], mis["y_pred"]])
except Exception as e:
    mis = None
    print("note: text lookup unavailable —", e)

rows=[]
for sg in SUBGROUPS_4:
    sg_all = pred[pred["subgroup"]==sg]
    sg_err = errors[errors["subgroup"]==sg]
    if len(sg_all)==0: continue
    for a in CLASSES:
        for pcls in CLASSES:
            if a==pcls: continue
            n = len(sg_err[(sg_err["y_true"]==a)&(sg_err["y_pred"]==pcls)])
            if n==0: continue
            ex = ""
            if mis is not None:
                cand = mis[(mis["subgroup"]==sg)&(mis["y_true"]==a)&(mis["y_pred"]==pcls)]
                if len(cand): ex = str(cand.sort_values("confidence",ascending=False).iloc[0]["text"])[:120]
            rows.append({
                "Error Type": f"{a.capitalize()} -> {pcls.capitalize()}",
                "Subgroup Mainly Affected": sg,
                "Actual Sentiment": a, "Predicted Sentiment": pcls,
                "Number of Cases": n,
                "Percentage of Total Errors": round(n/total*100,2),
                "Rate Within Subgroup": round(n/len(sg_all)*100,2),
                "Example Text": ex,
            })
table5 = pd.DataFrame(rows).sort_values("Number of Cases", ascending=False).reset_index(drop=True)
table5.to_csv(OUT/"Table5_complete.csv", index=False)
print(f"Total errors: {total}")
print(table5.head(20).to_string(index=False))


Total errors: 4937
          Error Type Subgroup Mainly Affected Actual Sentiment Predicted Sentiment  Number of Cases  Percentage of Total Errors  Rate Within Subgroup                                                                                                             Example Text
 Negative -> Neutral                   formal         negative             neutral             1491                       30.20                 14.34                                                    Melania Trump sues Slovenia journalist over 'escort' claims via @user
 Neutral -> Positive                   formal          neutral            positive              948                       19.20                  9.12                                     @user When will Melania do her "I have a dream" speech? I'm looking forward to it :)
 Neutral -> Negative                   formal          neutral            negative              855                       17.32                  8.22 Sheikh Othman #Ade

## Cell 8: Copy the already-complete novelty tables (6–13) into the assembled folder unchanged

In [8]:
import shutil
already = {
 "Table6_Ethical_Risk_Ranking.csv":"Table6_complete.csv",
 "Table7_Imbalance_Correction.csv":"Table7_complete.csv",
 "Table8_Model_Design.csv":"Table8_complete.csv",
 "Table9_Dual_Framework_Audit.csv":"Table9_complete.csv",
 "Table10_Confidence_Risk.csv":"Table10_complete.csv",
 "Table11_Fairness_Drift.csv":"Table11_complete.csv",
 "Table12_Error_Taxonomy.csv":"Table12_complete.csv",
 "Table13_Ensemble_Weighting.csv":"Table13_complete.csv",
}
for src, dst in already.items():
    s = PATHS["results"]/src
    if s.exists():
        shutil.copy(s, OUT/dst); print("copied", dst)
    else:
        print("MISSING:", src)


copied Table6_complete.csv
copied Table7_complete.csv
copied Table8_complete.csv
copied Table9_complete.csv
copied Table10_complete.csv
copied Table11_complete.csv
copied Table12_complete.csv
copied Table13_complete.csv


## Cell 9: TABLE 3 — Tuning (complete: fills every model row, no blanks)

In [9]:
# TABLE 3 — read the full five-model tuning table produced by NB7T.
# (NB7T tunes all five baselines on FS1 and saves Table3_Full_Tuning_All_Models.csv.)
# This keeps ALL tables in one assembled folder.
src = PATHS["results"] / "Table3_Full_Tuning_All_Models.csv"
if src.exists():
    table3 = pd.read_csv(src)
    table3.to_csv(OUT/"Table3_complete.csv", index=False)
    print("TABLE 3 — full five-model tuning (from NB7T)")
    show = [c for c in ["Model","Before Tuning Macro F1","After Tuning Macro F1",
                        "Macro F1 Improvement","Final Rank After Tuning"] if c in table3.columns]
    print(table3[show].to_string(index=False))
else:
    # NB7T not run yet — fall back to the single-model Exp 10 result so the folder is still complete
    t3 = pd.read_csv(PATHS["results"]/"Table3_Hyperparameter_Tuning.csv")
    lr = t3.iloc[0]
    table3 = pd.DataFrame([{
        "Model":"Logistic Regression","Dataset":"TweetEval","Feature Set Used":"Hybrid feature set",
        "Before Tuning Accuracy":lr.get("Before Tuning Accuracy",""),
        "Before Tuning Macro F1":lr["Before Tuning Macro F1"],
        "After Tuning Accuracy":lr.get("After Tuning Accuracy",""),
        "After Tuning Macro F1":lr["After Tuning Macro F1"],
        "Best Parameters":lr["Best Parameters"],
        "Macro F1 Improvement":lr["Macro F1 Improvement"],
        "Final Rank After Tuning":1,
        "Note":"NB7T not run yet — showing Exp 10 final-model tuning only. Run NB7T for the full five-model table."}])
    table3.to_csv(OUT/"Table3_complete.csv", index=False)
    print("TABLE 3 — single-model (NB7T not run yet). Run NB7T then re-run this cell for all five.")
    print(table3.to_string(index=False))


TABLE 3 — full five-model tuning (from NB7T)
             Model  Before Tuning Macro F1  After Tuning Macro F1  Macro F1 Improvement  Final Rank After Tuning
LogisticRegression                  0.5453                 0.5672                0.0219                      1.0
         LinearSVM                  0.5648                 0.5648                0.0000                      2.0
     MultinomialNB                  0.4349                 0.5589                0.1241                      3.0
           XGBoost                  0.4671                 0.5230                0.0559                      4.0
      RandomForest                  0.4169                 0.4169                0.0001                      5.0
 Gradient Boosting                     NaN                    NaN                   NaN                      NaN


## Cell 10: Done — everything assembled

`results/assembled/` now holds a complete CSV for every table, no blank computed
fields, plus the new per-class tables. Download the folder and paste each CSV into
its tab, or use these as the source of truth for the write-up.

In [10]:
print("ASSEMBLED FILES:")
for f in sorted(OUT.glob("*.csv")):
    df = pd.read_csv(f)
    print(f"  {f.name:32s}  {df.shape[0]} rows x {df.shape[1]} cols")
print("\nAll 13 tables assembled in one folder:")
print("  Table3 = full five-model tuning (from NB7T) if present, else single-model fallback.")
print("New analytical content: per-class (neg/neu/pos) metrics in Table1, Table1c, Table2, Table4.")
print("Design: four subgroups; sarcasm flagged exploratory (n=14); no 'mixed' group.")


ASSEMBLED FILES:
  Table10_complete.csv              5 rows x 11 cols
  Table11_complete.csv              5 rows x 12 cols
  Table12_complete.csv              4 rows x 10 cols
  Table13_complete.csv              3 rows x 13 cols
  Table1_complete.csv               5 rows x 23 cols
  Table1c_final_perclass.csv        3 rows x 6 cols
  Table2_complete.csv               4 rows x 15 cols
  Table3_complete.csv               6 rows x 14 cols
  Table4_complete.csv               4 rows x 20 cols
  Table5_complete.csv               21 rows x 8 cols
  Table6_complete.csv               1 rows x 17 cols
  Table7_complete.csv               4 rows x 16 cols
  Table8_complete.csv               2 rows x 14 cols
  Table9_complete.csv               3 rows x 12 cols

All 13 tables assembled in one folder:
  Table3 = full five-model tuning (from NB7T) if present, else single-model fallback.
New analytical content: per-class (neg/neu/pos) metrics in Table1, Table1c, Table2, Table4.
Design: four subgroups; 